# DreamerV3 one-step latent transition -- the sampling-to-proof audit (D1)

Measures the **sampling-to-proof gap** on a *frozen* DreamerV3 `size1m` model:
latent states where a large random-sampling audit of a Lyapunov decrease
condition finds nothing, but CROWN branch-and-bound finds a genuine violation
**that the model's own dynamics actually reach**.

### The certified object (locked scope)

The **one-step** latent map under the **frozen deterministic actor**,

    z = (deter, stoch)  ->  z' = (deter', stoch')

where deter' = RSSM core(z, pi(z)), stoch' = the MEAN of the prior
categorical, and pi(z) = tanh(mean(actor)). Nothing about the imagined
rollout, and nothing larger than `size1m` (deter 512, hidden 64, stoch 32x4).
The model is audited as trained; only the Lyapunov candidate V moves.

### What this run will and will not claim

- It reports the numbers **this run** measures, next to the model, region,
  and condition that produced them. No figure is carried over from any other
  system.
- The verifier verdict is **three-way**: `violation` / `unknown` / `certified`.
  `unknown` means the bound stayed loose and no counterexample was found. That
  is verifier incompleteness and is **never** evidence of safety, and it is
  **never** a gap and **never** a partial finding.
- `gap_demonstrated` is True only when the sampling audit found NOTHING and
  branch-and-bound found at least one violation at a latent state the model
  demonstrably reaches (the **reachability gate**). An off-distribution
  violation is discarded, not reported with a caveat.
- The certified graph contains the categorical output stoch' = exp/sum-div.
  auto_LiRPA cannot interval-bind that at any useful width (measured in
  `d1_smoke.py`; the guardrail-predicted categorical wall). The audit probes
  this ONCE and records every branch-and-bound box as `unknown` with the
  reason -- never certified, never a violation. The **sampling audit is the
  empirical baseline**, and a sampling violation is the only thing that can
  become a result; it must survive the reachability gate first.
- The region is built FROM the model's own on-policy latent states (real
  posterior rollouts). The certified map is the prior-mean surrogate. That
  distinction is recorded in the artifact, never conflated.

## 1. Environment

Everything is pinned. The dreamerv3 repo at commit `e3f0224` VENDORS the
`embodied` library it trains with -- that vendored `nets.py` is the exact
math the torch port (`src/dreamer.py`) transcribes, so training with the
pinned commit makes the port and the trained model the same function by
construction. `rl-wm-audit` itself is **not published**: upload a zip of the
working tree (primary path). `cnl-work` is published; a clone is fine, and
the verifier is pinned by the SHA-256 of `src/verify.py`.

In [ ]:
REPO_URL       = 'https://github.com/sehajr-singhs/rl-wm-audit'
CNL_URL        = 'https://github.com/sehajr-singhs/certified-neural-lyapunov'
DREAMERV3_URL  = 'https://github.com/danijar/dreamerv3'
DREAMERV3_COMMIT = 'e3f02248693a79dc8b0ebd62c93683888ddaccfe'
VERIFY_SHA256  = '8c1505c4a06bf17760ce250cdabdb023f4f4ab71772e0a732a7f4d655d097555'

import os, subprocess, zipfile

# --- rl-wm-audit: zip upload is the primary path --------------------------
if os.path.isdir('/content/rl-wm-audit'):
    print('present: /content/rl-wm-audit')
else:
    from google.colab import files
    print('Upload rl-wm-audit.zip (repo is unpublished; clone will not work).')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('/content/')
    print('unpacked:', name)

assert os.path.isdir('/content/rl-wm-audit'), 'repo not present; upload the zip'

# --- dreamerv3 at the pinned commit ----------------------------------------
if not os.path.isdir('/content/dreamerv3'):
    r = subprocess.run(['git', 'clone', DREAMERV3_URL, '/content/dreamerv3'],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-500:]
    r = subprocess.run(['git', '-C', '/content/dreamerv3', 'checkout',
                        DREAMERV3_COMMIT], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-500:]
got = subprocess.run(['git', '-C', '/content/dreamerv3', 'rev-parse', 'HEAD'],
                     capture_output=True, text=True).stdout.strip()
assert got == DREAMERV3_COMMIT, f'dreamerv3 HEAD {got} != {DREAMERV3_COMMIT}'
print('dreamerv3 pinned at', got[:12])

# --- cnl-work: published, so clone is fine here ----------------------------
if not os.path.isdir('/content/cnl-work'):
    r = subprocess.run(['git', 'clone', '--depth', '1', CNL_URL, '/content/cnl-work'],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-500:]
os.environ['CNL_WORK'] = '/content/cnl-work'
os.environ['MUJOCO_GL'] = 'egl'
%cd /content/rl-wm-audit

### 1b. Verifier provenance, pinned by content

The verifier commit this project was validated against is `6ccf7ae`, which is
**local-only** and ahead of the published `cnl-work` HEAD, so a clone reports
a different commit even though `src/verify.py` is byte-identical. The pin is
therefore the **SHA-256 of the verifier source**, the thing that actually
determines the result. If this assertion fails, stop: nothing below may be
reported.

In [ ]:
import hashlib

vp = '/content/cnl-work/src/verify.py'
digest = hashlib.sha256(open(vp, 'rb').read()).hexdigest()
print('verify.py sha:', digest[:16], '...')
assert digest == VERIFY_SHA256, (
    'verify.py does NOT match the validated verifier.\\n'
    'Stop here. Do not report results from an unverified driver.')
print('OK: byte-identical to the validated verifier.')

## 2. Install

dreamerv3 pins `jax[cuda12]==0.4.33` and `numpy<2`. `dm_control` is the
environment suite (headless via `MUJOCO_GL=egl`), and `auto_LiRPA` is the
bounder. Colab already ships torch.

**`auto_LiRPA` is NOT installed from PyPI.** The published PyPI releases
carry legacy `torch<1.13` constraints that pip cannot resolve against
Colab's torch 2.11 (measured: `ResolutionImpossible` and `Could not find a
version that satisfies torch<1.13.0,>=1.8.0`). The verifier this project
was validated against is auto_LiRPA **0.7.2** at GitHub commit
`5a098e8f` (June 2026 release; requires `torch>=2.0,<2.12`, satisfied by
Colab's torch). Its metadata declares `python_requires='~=3.11.0'`, but
that is advisory: the identical commit runs on Python 3.13 on the
validating machine, so the notebook installs it with
`--ignore-requires-python` on Colab's Python 3.12 and the correctness
gates re-validate it here. Install order matters: dreamerv3's `numpy<2`
is applied first, then auto_LiRPA's `numpy>=2` bump leaves numpy 2.x --
the exact numpy/torch/auto_LiRPA combination the D1 work was validated on.

**Runtime: GPU is required** (T4 or better). The audit itself is small, but
training size1m from pixels on CPU is not practical.

In [ ]:
# --- pinned installs (fail loudly: no | tail to mask exit codes) ---------
!pip -q install -U -r /content/dreamerv3/requirements.txt
!pip -q install dm_control
# verifier: auto_LiRPA 0.7.2 @ 5a098e8f from GitHub, NOT PyPI (PyPI releases
# carry torch<1.13 and pip cannot resolve them against modern torch). Its
# torch>=2.0,<2.12 range is satisfied by Colab's torch; its numpy>=2 bump
# after the numpy<2 downgrade above is the validated numpy 2.x combo.
# --ignore-requires-python: 5a098e8f's metadata says ~=3.11.0 but that is
# advisory -- the same commit runs on Python 3.13 on the validating machine.
# The correctness gates re-validate the verifier on Colab's Python 3.12.
!pip -q install 'git+https://github.com/Verified-Intelligence/auto_LiRPA.git@5a098e8f9fb5786a428a024981d833d303921f2d' --ignore-requires-python
!pip -q install ruamel.yaml

import importlib
for m in ['torch', 'auto_LiRPA', 'numpy', 'jax', 'dm_control', 'ninjax', 'elements']:
    mod = importlib.import_module(m)
    print(f'{m:16s}', getattr(mod, '__version__', 'ok'))

import auto_LiRPA
assert auto_LiRPA.__version__ == '0.7.2', (
    f'verifier mismatch: auto_LiRPA {auto_LiRPA.__version__} (want 0.7.2 @ 5a098e8f); '
    'do not proceed with an unvalidated verifier')
print('OK: auto_LiRPA 0.7.2 @ 5a098e8f (the validated verifier)')

In [ ]:
import jax, torch

print('jax devices   :', jax.device_count(), jax.devices())
print('torch cuda    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert (jax.device_count() >= 1 and torch.cuda.is_available()), (
    'GPU runtime required (Runtime -> Change runtime type -> T4 GPU)')

import dm_control  # noqa: F401
from dm_control import suite  # noqa: F401
env = suite.load('cartpole', 'balance')
env.reset()
print('dm_control cartpole_balance OK')

## 3. Correctness gates

Two suites. Each covers a failure that would silently invalidate every number
downstream, and each was added because something in it actually bit.

| suite | covers |
|---|---|
| `d1_smoke.py` | that auto_LiRPA traces and CROWN-bounds the full one-step graph, and that the bounds are SOUND; it also MEASURES the looseness (~1e6x at r=0.02), the scaling boundary that makes branch-and-bound return unknown on realistic boxes |
| `test_dreamer_transition.py` | transcription: the torch port vs an independent re-implementation of the upstream formulas, plus the JAX-checkpoint loader end to end |

**If any gate fails, stop.** Do not interpret results past a failed gate.

In [ ]:
!python experiments/d1_smoke.py
print()
!python tests/test_dreamer_transition.py

## 4. Train the frozen model

Train DreamerV3 `size1m` on **cartpole_balance from pixels** (`dmc_vision`
config: image observation, proprioception off). The task is irrelevant to the
claim -- we audit the trained model's latent dynamics; we do not benchmark
the model. `size1m` is the smallest official config and the locked scope.

Colab-specific overrides: `--replay.size 200000` (the paper's 5e6 of raw
images would not fit in RAM) and `--jax.prealloc false` (T4 memory).

**Runtime**: roughly 1.5-3 h for 1e6 steps on a T4. Lower `STEPS` (e.g.
300000) for a quicker but weaker model; the audit's region and claim adapt
to whatever was trained. Checkpoints are saved every 5 minutes, and a rerun
with the same `LOGDIR` resumes. If a checkpoint already exists the training
cell is skipped.

In [ ]:
import os, subprocess, time, pathlib

LOGDIR  = '/content/d1_logdir'
STEPS   = 1_000_000       # env steps; lower to 300_000 for a quick run
TRAIN_RATIO = 256          # inherited from dmc_vision; leave alone

cmd = ['python', 'dreamerv3/main.py',
       '--logdir', LOGDIR,
       '--configs', 'defaults', 'dmc_vision',
       '--task', 'dmc_cartpole_balance',
       '--run.steps', str(STEPS),
       '--run.train_ratio', str(TRAIN_RATIO),
       '--run.save_every', '300',
       '--replay.size', '200000',
       '--jax.prealloc', 'false',
       '--jax.compute_dtype', 'bfloat16',
       '--seed', '0']
print(' '.join(cmd))

ckpt_latest = pathlib.Path(LOGDIR) / 'ckpt' / 'latest'
if ckpt_latest.exists():
    print('checkpoint exists -> skipping training (resume by rerunning)')
else:
    pathlib.Path(LOGDIR).mkdir(parents=True, exist_ok=True)
    log = open('/content/d1_train.log', 'w')
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    p.wait()
    log.close()
    print(f'training exited rc={p.returncode} after {time.time()-t0:.0f}s')
    print('--- last log lines ---')
    print('\n'.join(open('/content/d1_train.log').read().splitlines()[-15:]))

# summary from the metrics file, if present
mfile = pathlib.Path(LOGDIR) / 'scores.jsonl'
if mfile.exists():
    import json as _json
    rows = [_json.loads(l) for l in mfile.read_text().splitlines() if l.strip()]
    if rows:
        for key in ('episode/score', 'episode/length', 'train/loss/model'):
            vals = [r.get(key) for r in rows if key in r]
            if vals:
                print(key, 'last=%.3g max=%.3g n=%d' % (vals[-1], max(vals), len(vals)))
assert ckpt_latest.exists(), 'no checkpoint after training; inspect /content/d1_train.log'

## 5. Export the weights under the upstream key names

The embodied checkpoint stores the params as `{'params': {key: array}}` with
ninjax flat keys (`dyn/dynin0/kernel`, `pol/mlp/linear0/kernel`, ...). We
read it directly and export exactly the two modules the audit needs,
**under the upstream key names**, so `load_jax_arrays` matches them by
suffix:

    dyn/*   ->  OneStepLatentTransition (RSSM core + prior)
    pol/*   ->  PolicyMean (actor, deterministic tanh(mean))

A sidecar JSON records the pins and the resolved config for the artifact.

In [ ]:
import json, pathlib, pickle
import numpy as np
import elements

logdir  = elements.Path(LOGDIR)
ckpt_dir = logdir / 'ckpt'
latest  = (ckpt_dir / 'latest').read_text().strip()
data    = pickle.loads((ckpt_dir / latest / 'agent.pkl').read_bytes())
params  = data['params']
print('param keys:', len(params), '| counters:', data['counters'])

dyn = {k: np.asarray(v, np.float32) for k, v in params.items() if k.startswith('dyn/')}
pol = {k: np.asarray(v, np.float32) for k, v in params.items() if k.startswith('pol/')}
print('dyn keys:', len(dyn), 'pol keys:', len(pol))

# the exact keys the port's loader must find
expected = ['dyn/dynin0/kernel', 'dyn/dynin0/bias', 'dyn/dynin1/kernel',
            'dyn/dynhid0/kernel', 'dyn/dyngru/kernel',
            'dyn/prior0/kernel', 'dyn/priorlogit/kernel',
            'pol/mlp/linear0/kernel', 'pol/head/mean/kernel']
missing = [k for k in expected if k not in dyn and k not in pol]
assert not missing, f'missing expected keys: {missing}'

outdir = '/content/rl-wm-audit/artifacts'
os.makedirs(outdir, exist_ok=True)
np.savez(os.path.join(outdir, 'd1_weights.npz'), **{**dyn, **pol})

# provenance sidecar
sidecar = dict(
    upstream_dreamerv3='github.com/danijar/dreamerv3 @ ' + DREAMERV3_COMMIT,
    upstream_embodied=('github.com/danijar/embodied @ a6583c14123ad310a3f20a6e78b5a0983d24a4fb (vendored in dreamerv3@e3f0224, verified against the port transcription)'),
    counters=data['counters'],
    n_dyn_params=sum(int(v.size) for v in dyn.values()),
    n_pol_params=sum(int(v.size) for v in pol.values()),
    sizes={k: list(v.shape) for k, v in {**dyn, **pol}.items() if v.ndim >= 2},
)
json.dump(sidecar, open(os.path.join(outdir, 'd1_weights_meta.json'), 'w'), indent=2)
print('exported', os.path.join(outdir, 'd1_weights.npz'))
print('dynin0 kernel', dyn['dyn/dynin0/kernel'].shape,
      '| priorlogit kernel', dyn['dyn/priorlogit/kernel'].shape)

## 6. On-policy latent support (real rollouts)

The region and the reachability gate need the latent states the model ACTUALLY
visits: `(deter_t, stoch_t)` from real episodes, where stoch_t is the
ENCODER posterior (the model's true latent), not the prior mean. The audit's
certified map is the prior-mean surrogate; the region lives where the real
model lives. That distinction is recorded in the result artifact.

The trained agent is rebuilt in-process (same config, `--jax.precompile
false` to skip the train-graph compile), the checkpoint is loaded with the
same `elements.Checkpoint` the trainer used, and `agent.policy` drives the
wrapped dm_control environment. The visited latent is read from the carry
after each policy call. The first policy call compiles the jitted graph
(~1-2 min); then it is fast.

Defaults: 200 episodes x up to 250 steps = ~50k latents (~10-20 min).
Raise `N_EPISODES` for a denser region.

In [ ]:
import numpy as np
import elements
import ruamel.yaml as yaml
from dreamerv3.main import make_agent, make_env

N_EPISODES = 200
HORIZON    = 250

# rebuild the config exactly as trained, then load the trained params
logdir = elements.Path(LOGDIR)
config = elements.Config(yaml.YAML(typ='safe').load((logdir / 'config.yaml').read()))
config = elements.Flags(config).parse(['--jax.precompile', 'false'])

agent = make_agent(config)
cp = elements.Checkpoint(logdir / 'ckpt')
cp.agent = agent
cp.load()
env = make_env(config, 0)
print('agent rebuilt and checkpoint loaded; params:', len(agent.params))

deter_all, stoch_all = [], []
carry = agent.init_policy(1)
for ep in range(N_EPISODES):
    obs = env.reset()
    carry = agent.init_policy(1)
    for t in range(HORIZON):
        carry, act, _ = agent.policy(carry, obs, mode='eval')
        dc = carry[1]                                  # dyn carry: visited latent
        deter_all.append(np.asarray(dc['deter'], np.float32)[0])
        stoch_all.append(np.asarray(dc['stoch'], np.float32)[0])
        act1 = {k: np.asarray(v, np.float32)[0] for k, v in act.items()}
        obs = env.step(act1)
        if obs['is_last']:
            break
    if (ep + 1) % 50 == 0:
        print(f'episode {ep+1}/{N_EPISODES}')

deter = np.stack(deter_all).astype(np.float32)
stoch = np.stack(stoch_all).astype(np.float32)
print('latents:', deter.shape, stoch.shape)
assert deter.shape == (len(deter_all), 512) and stoch.shape == (len(deter_all), 32, 4)
assert np.isfinite(deter).all() and np.isfinite(stoch).all()

out = '/content/rl-wm-audit/artifacts/d1_support.npz'
np.savez(out, deter=deter, stoch=stoch,
         n_episodes=N_EPISODES, horizon=HORIZON,
         upstream_dreamerv3=DREAMERV3_COMMIT)
print('wrote', out)

## 7. Load check (fast)

Prove in seconds that the exported checkpoint loads into the torch port and
that the closed loop is finite -- before spending the audit's compute. A
failure here means the export or the loader contract is wrong, and nothing
below may be reported.

In [ ]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
sys.path.insert(0, '/content/rl-wm-audit')
import numpy as np
import torch
from src.dreamer import (OneStepLatentTransition, PolicyMean,
                         OneStepClosedLoop, RSSMConfig, load_jax_arrays)

cfg = RSSMConfig()
trans = OneStepLatentTransition(cfg, action_dim=1).eval()
pol = PolicyMean(cfg.deter, cfg.stoch, cfg.classes, act_dim=1).eval()
w = np.load('/content/rl-wm-audit/artifacts/d1_weights.npz')
load_jax_arrays(trans, {k: w[k] for k in w.files})
load_jax_arrays(pol, {k: w[k] for k in w.files})
loop = OneStepClosedLoop(trans, pol).eval()

z = torch.cat([torch.zeros(1, 512), torch.full((1, 128), 0.25)], -1)
with torch.no_grad():
    z2 = loop(z)
print('closed loop forward OK; z2', tuple(z2.shape),
      'finite:', bool(torch.isfinite(z2).all()))
assert z2.shape == (1, 640) and torch.isfinite(z2).all()

# the deterministic actor stays inside [-1, 1]
a = pol(torch.zeros(1, 512), torch.full((1, 32, 4), 0.25))
print('actor output range:', float(a.min()), float(a.max()))
assert float(a.abs().max()) <= 1.0 + 1e-5

## 8. The audit

Runs `experiments/d1_sampling_gap.py`, mirroring A1 on the latent space:

    frozen model -> latent fixed point z* (settled and VERIFIED) ->
    region from on-policy latents minus a k-dim annulus about z* ->
    fit V -> sampling audit (500k) -> branch-and-bound (wall-gated) ->
    reachability gate -> gap_demonstrated

Runtime on a T4 GPU: roughly 10-20 min (V training dominates; the
branch-and-bound probe's BoundedModule build is CPU-bound).

**Reading the result, whatever it says:**

- `sampling.n_violations == 0` is an empirical baseline, not a certificate.
- Every `unknown` box is category-3 verifier incompleteness (expected: the
  categorical-output wall), never a gap, never a finding.
- `gap_demonstrated` is True only if BaB found a violation sampling missed AND
  the model reaches it. That is the only outcome that would be a result.

In [ ]:
import json, os, subprocess

cmd = ['python', 'experiments/d1_sampling_gap.py',
       '--weights',  '/content/rl-wm-audit/artifacts/d1_weights.npz',
       '--support',  '/content/rl-wm-audit/artifacts/d1_support.npz',
       '--action-dim', '1',
       '--n-samples', '500000',
       '--v-steps', '4000',
       '--v-batch', '4096',
       '--hole-k', '4',
       '--burn-in', '100',
       '--time-budget', '120',
       '--method', 'CROWN']
print(' '.join(cmd))
log = open('/content/d1_audit.log', 'w')
p = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, cwd='/content/rl-wm-audit')
log.close()
print('\n'.join(open('/content/d1_audit.log').read().splitlines()[-60:]))
assert p.returncode == 0, 'audit failed; see /content/d1_audit.log'

## 9. The artifact and its honest reading

The result JSON is the artifact. Keep it, quote only it, and read it with
the rules of section 1: `unknown` is never a gap, and `gap_demonstrated` is
the only flag that means anything happened. If it is False, the honest
one-line summary is: *no sampling-to-proof gap on this policy/region; the
categorical output is the documented verifier boundary (category 3).*

In [ ]:
import json, pathlib

res = json.load(open('/content/rl-wm-audit/results/d1_seed0.json'))
r = res['result']
print('sampling        :', f"{r['sampling_violations']}/{res['sampling']['n_sampled']}",
      f"violations, worst cond {res['sampling']['worst_cond']:.3e}")
print('bab verdicts    :',
      f"{res['bab']['n_certified']} certified, {res['bab']['n_violation']} violation,",
      f"{res['bab']['n_unknown']} unknown")
if res['bab'].get('wall'):
    print('bab wall        :', res['bab']['wall'][:120], '...')
print('reachable CEs   :', r['reachable_counterexamples'])
print('gap_demonstrated:', r['gap_demonstrated'])
print()
print(r['interpretation'])

from google.colab import files
files.download('/content/rl-wm-audit/results/d1_seed0.json')
files.download('/content/rl-wm-audit/artifacts/d1_weights_meta.json')